# Compare the saved ear-pressure adapter with base Llama
The default question tests pituitary function, a topic outside the two pilot training examples. Select a GPU, enable notebook access to HF_TOKEN, and run all cells. Choose the completed ear-pressure run if prompted. Both models receive identical source excerpts and generation settings for each audience.

This is a Buddy chat comparison with supplied evidence. It tests answer generation. Search retrieval and the six-deck browser workflow need separate tests. The pituitary question is also in the lab’s draft benchmark (HBM-002). Keep that in mind if you use these results to tune the model.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
SOURCE_COMMIT = '189285c8b31a102b61038834f7f00928102de5cc'
REPO = Path('/content') / ('healthbrowse-pilot-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--filter=blob:none', '--branch', 'research/healthbrowse-medical-model',
                    'https://github.com/McChandra/medical_chatbot.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('The pilot checkout does not match the pinned source commit.')
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Pinned pilot source:', head)


In [ ]:
# Keep the text-model stack independent of packages preinstalled in the Colab kernel.
import venv
ENV_DIR = Path('/content') / ('healthbrowse-train-' + sys.implementation.cache_tag)
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
LAB_PYTHON = str(ENV_DIR / 'bin/python')
lab_env = os.environ.copy()
lab_env.pop('PYTHONPATH', None)
lab_env['PYTHONNOUSERSITE'] = '1'

def run_lab(*args):
    subprocess.run([LAB_PYTHON, *map(str, args)], cwd=LAB, env=lab_env, check=True)

# Target the venv directly; Colab's system Python may not ship ensurepip.
subprocess.run([sys.executable, '-m', 'pip', '--python', LAB_PYTHON,
                'install', '-e', '.[train,test]'], cwd=LAB, env=lab_env, check=True)
run_lab('-c', """import torch, transformers, accelerate, bitsandbytes
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.integrations import get_reporting_integration_callbacks
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Trainer imports passed.', 'torch', torch.__version__, 'transformers', transformers.__version__)
""")


In [ ]:
from google.colab import userdata
run_lab('-m', 'hb_model_lab.cli', 'preflight')
run_lab('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before training.'")
try:
    lab_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Enable notebook access to HF_TOKEN in Colab Secrets.") from exc


In [ ]:
TOPIC = "pituitary"  # @param ["pituitary", "ear-pressure"]
from google.colab import drive
drive.mount('/content/drive')
base_dir = Path('/content/drive/MyDrive/HealthBrowseModelLab')
runs = sorted(p for p in base_dir.glob('ear-pressure-*') if (p / 'model/adapter/adapter_model.safetensors').is_file())
if not runs:
    raise FileNotFoundError(f'No completed ear-pressure run found in {base_dir}. Mount the Drive account used for training.')
if len(runs) == 1:
    run = runs[0]
else:
    print('Completed runs:', ', '.join(p.name for p in runs))
    selected = input('Enter the exact folder name: ').strip()
    run = base_dir / selected
    if run not in runs:
        raise ValueError('Select a folder from the list.')
report = run / (TOPIC + '-base-vs-adapter-' + uuid.uuid4().hex[:8] + '.json')
print('Evaluating', run.name)


In [ ]:
experiment = r"""
import gzip
import json
import sys
from pathlib import Path

import torch
from peft import PeftModel
from hb_model_lab.contracts import Query, validate_answer
from hb_model_lab.data import digest, read_jsonl
from hb_model_lab.prompts import messages
from hb_model_lab.runtime import ModelRunner, adapter_manifest
from hb_model_lab.training import load_config

root, destination = map(Path, sys.argv[1:3])
topic = sys.argv[3]
config = root / 'pilot-config.json'
adapter = root / 'model/adapter'
manifest = adapter_manifest(adapter, load_config(config), allow_smoke=True)
data = root / 'data/train.jsonl'
if digest(data) != manifest['train_sha256']:
    raise ValueError('The saved examples differ from the training manifest.')
rows = read_jsonl(data)
if len(rows) != 2 or {r['topic_group'] for r in rows} != {'middle-ear-pressure'}:
    raise ValueError('Select the two-example ear-pressure pilot.')
if topic == 'pituitary':
    question = 'What is the main function of the pituitary gland?'
    ids = ['m46699-fs-id2955108', 'm46699-fs-id2161906']
    with gzip.open('data/education-corpus.jsonl.gz', 'rt', encoding='utf-8') as f:
        passages = {record['id']: record for record in map(json.loads, f) if record['id'] in ids}
    evidence = [{k: passages[i][k] for k in ('id','title','url','text')} for i in ids]
    validation = root / 'data/validation.jsonl'
    if digest(validation) != manifest['validation_sha256']:
        raise ValueError('Validation file differs from the training manifest.')
    used = rows + read_jsonl(validation)
    used_pages = {e['url'] for r in used for e in r['input']['evidence']}
    if used_pages.intersection(e['url'] for e in evidence):
        raise ValueError('The test source page occurred in the pilot data.')
else:
    if topic != 'ear-pressure':
        raise ValueError('Unknown comparison topic.')
    question = 'Why might my ears pop when I yawn?'
    evidence = rows[0]['input']['evidence']
    assert all(r['input']['evidence'] == evidence for r in rows)
queries = [Query(question=question, role=r, mode='chat', topic_kind='knowledge',
                 evidence=evidence) for r in ('general', 'clinician')]
runner = ModelRunner(config, baseline=True)
tokenizer = runner.tokenizer
results = []
for condition in ('Base Llama', 'Pilot adapter'):
    if condition == 'Pilot adapter':
        runner.model = PeftModel.from_pretrained(runner.model, str(adapter), is_trainable=False)
        runner.model.eval()
    for query in queries:
        prompt = tokenizer.apply_chat_template(messages(query), tokenize=True,
                  add_generation_prompt=True, return_tensors='pt', return_dict=False).to(runner.model.device)
        with torch.inference_mode():
            output = runner.model.generate(prompt, attention_mask=torch.ones_like(prompt),
                      max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id,
                      eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')])
        raw = tokenizer.decode(output[0, prompt.shape[1]:], skip_special_tokens=True)
        record = dict(condition=condition, role=query.role, raw=raw)
        try:
            answer = validate_answer(json.loads(raw), query)
            record.update(status=answer.status, validated=answer.model_dump())
        except ValueError as exc:
            record.update(status='format or citation check failed', error=str(exc))
        results.append(record)
        print(condition, query.role, record['status'], flush=True)
with destination.open('x') as f:
    json.dump(dict(question=question, evidence=evidence, adapter_sha256=manifest['adapter_sha256'],
                   max_new_tokens=512, results=results), f, indent=2, ensure_ascii=False)
"""
run_lab('-c', experiment, run, report, TOPIC)


In [ ]:
import html
from IPython.display import HTML, display
result = json.loads(report.read_text())
print('Question:', result['question'])
for e in result['evidence']:
    print(e['id'], e['title'], '\n' + e['text'] + '\n')
def show_record(r):
    if 'validated' not in r:
        return html.escape(r['status'] + '\n' + r['raw'])
    a = r['validated']
    lines = [claim['text'] for claim in a['answer']] or [a['status'], a['message'], *a['choices']]
    return html.escape('\n'.join(lines))
records = {(r['role'], r['condition']): r for r in result['results']}
table = '<table><tr><th>Audience</th><th>Base Llama</th><th>Pilot adapter</th></tr>'
for role in ['general','clinician']:
    table += '<tr><th>' + role.title() + '</th>'
    for model in ['Base Llama','Pilot adapter']:
        table += '<td style="white-space:pre-wrap;vertical-align:top">' + show_record(records[role,model]) + '</td>'
    table += '</tr>'
display(HTML(table + '</table>'))
print('Saved comparison:', report)
